# Socle métadonnées-driven — découverte, sérialisation, règles métier

Ce notebook démontre le socle transverse **`MyIA.AI.Shared`** (.NET 9.0) sur les trois
moments qui le fondent, en partant d'un domaine factice (facturation / catalogue) :

1. **Décoration → introspection.** Des attributs (`[MainCategory]`, `[AttributeContainer]`)
   rendent des types découvrables *sans aucun appel d'enregistrement* — un
   `ReflectedProviderContainer.FromAssembly<T>()` les regroupe par catégorie et par rôle.
2. **Sérialisation round-trip.** Le même graphe d'entités (hiérarchie `IChildEntity`
   sur plusieurs niveaux) part et revient intact en JSON **et** en XML, parent re-lié.
3. **Prédicat universel métier (Flee).** Une règle métier est une **chaîne** — venue
   d'un fichier, d'un CSV, d'un utilisateur non-développeur — compilée une fois puis
   appliquée à N instances. C'est la différence entre coder N branches `if` et piloter
   la logique métier par les données.

> **Prérequis.** Le socle doit être compilé en Release à la racine du dépôt :
> `dotnet build MyIA.AI.Shared/MyIA.AI.Shared.csproj -c Release`.


In [1]:
// Setup : on charge le socle (DLL Release locale) + Flee (moteur de règles) + Newtonsoft.
#r "../../../MyIA.AI.Shared/bin/Release/net9.0/MyIA.AI.Shared.dll"
#r "nuget:Flee"
#r "nuget:Newtonsoft.Json"

using System;
using System.Collections.Generic;
using System.Linq;
using MyIA.AI.ComponentModel.Attributes;
using MyIA.AI.ComponentModel.Entities;
using MyIA.AI.ComponentModel.Providers;
using MyIA.AI.ComponentModel.Serialization;
using MyIA.AI.ComponentModel.Rules;
using Flee.PublicTypes;
using Newtonsoft.Json;

// Sanity check : les trois références sont résolues. Flee reste chargé car
// FleePredicateBuilder (Moment 3) s'appuie sur ce moteur en interne.
Console.WriteLine($"socle    : {typeof(ReflectedProviderContainer).Assembly.GetName().Version}");
Console.WriteLine($"Flee     : {typeof(ExpressionContext).Assembly.GetName().Version}");
Console.WriteLine($"Newtonsoft: {typeof(JsonConvert).Assembly.GetName().Version}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Flee, 2.0.0 Newtonsoft.Json, 13.0.4

socle    : 1.0.0.0


Flee     : 1.0.0.0


Newtonsoft: 13.0.0.0


## Moment 1 — Décoration → introspection

La décoration est le socle d'une architecture *métadonnées-driven* : on **déclare** ce
qu'une classe est (sa catégorie, son rôle hiérarchique) via des attributs, puis un
conteneur de réflexion **découvre** ces déclarations sans qu'aucun code métier n'ait à
appeler un `Register(...)`.

Définissons un domaine factice : un abonnement facturable (feuille plate `ISimpleEntity`)
et un catalogue `Catalog > Product > Variant` (hiérarchie `IChildEntity` sur **3 niveaux**).


In [2]:
// Domaine : 4 entités decorées. Aucun Register() — tout passe par les attributs.

[MainCategory("Billing")]
public sealed class Subscription : ISimpleEntity
{
    public string Name { get; set; } = "";
    public double Montant { get; set; }       // en euros
    public string Pays { get; set; } = "";    // code pays ISO 2
    public bool Actif { get; set; }
}

[MainCategory("Catalog")]
[AttributeContainer(ChildType = typeof(Product))]
public sealed class Catalog : IChildEntity
{
    public string Nom { get; set; } = "";
    public IChildEntity? Parent { get; set; }
    private readonly List<IChildEntity> _children = new();
    public IReadOnlyList<IChildEntity> Children => _children;
    public void AddChild(IChildEntity child) { child.Parent = this; _children.Add(child); }
}

[MainCategory("Catalog")]
[AttributeContainer(ChildType = typeof(Variant))]
public sealed class Product : IChildEntity
{
    public string Nom { get; set; } = "";
    public double Prix { get; set; }
    public IChildEntity? Parent { get; set; }
    private readonly List<IChildEntity> _children = new();
    public IReadOnlyList<IChildEntity> Children => _children;
    public void AddChild(IChildEntity child) { child.Parent = this; _children.Add(child); }
}

[MainCategory("Catalog")]
public sealed class Variant : IChildEntity, IMergeable
{
    public string Nom { get; set; } = "";
    public double Prix { get; set; }
    public IChildEntity? Parent { get; set; }
    public IReadOnlyList<IChildEntity> Children => Array.Empty<IChildEntity>();
    public bool Merge(IMergeable other)
    {
        if (other is Variant v && string.IsNullOrEmpty(Nom)) { Nom = v.Nom; return true; }
        return false;
    }
}

Console.WriteLine("4 entites definies : Subscription (feuille), Catalog/Product/Variant (hiérarchie).");


4 entites definies : Subscription (feuille), Catalog/Product/Variant (hiérarchie).



(17,24): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(29,24): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(40,24): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



In [3]:
// Decouverte : FromAssembly<Subscription>() scanne l'assembly courant et regroupe.

var container = ReflectedProviderContainer.FromAssembly<Subscription>();

Console.WriteLine("Catégories découvertes (sans aucun Register) :");
foreach (var cat in container.Categories)
    Console.WriteLine($"  [{cat}]  ->  {string.Join(", ", container[cat].Select(t => t.Name))}");

Console.WriteLine();
Console.WriteLine($"Conteneurs (racines légitimes) : {string.Join(", ", container.Containers.Select(t => t.Name))}");
Console.WriteLine($"Entités hiérarchiques IChildEntity : {string.Join(", ", container.ChildEntities.Select(t => t.Name))}");
Console.WriteLine($"Entités plates ISimpleEntity : {string.Join(", ", container.SimpleEntities.Select(t => t.Name))}");
Console.WriteLine($"Fusibles IMergeable : {string.Join(", ", container.Mergeables.Select(t => t.Name))}");


Catégories découvertes (sans aucun Register) :


  [Billing]  ->  Subscription


  [Catalog]  ->  Catalog, Product, Variant


Conteneurs (racines légitimes) : Catalog, Product


Entités hiérarchiques IChildEntity : Catalog, Product, Variant


Entités plates ISimpleEntity : Subscription


Fusibles IMergeable : Variant


### Exercice 1 — Enregistrement explicite

`ReflectedProviderContainer` découvre par **décoration**. Le socle expose aussi
`SimpleProviderContainer`, symétrique par **enregistrement explicite** : on déclare
soi-même quel type va dans quelle catégorie, sans aucun attribut.

**Objectif.** Compléter la fonction `ConstruireConteneur()` pour qu'elle renvoie un
`SimpleProviderContainer` où `Subscription` est enregistré sous la catégorie `"Billing"`
et `Catalog` + `Product` sous `"Catalog"`. Les deux stratégies étant interchangeables via
`IProviderContainer`, le conteneur retourné doit exposer les mêmes `Categories` que le
conteneur par réflexion.

*Indice.* Le chaînage est fluent : `new SimpleProviderContainer().Register<T>("cat")...`.


In [4]:
// Exercice 1 (a completer) : enregistrement explicite, même lecture que par réflexion.
public IProviderContainer? ConstruireConteneur()
{
    // TODO etudiant : renvoyer un SimpleProviderContainer enregistrant
    //   Subscription  -> "Billing"
    //   Catalog       -> "Catalog"
    //   Product       -> "Catalog"
    return null;  // TODO etudiant
}

// --- zone de validation (ne pas modifier) ---
var explicite = ConstruireConteneur();
if (explicite is null)
{
    Console.WriteLine("Exercice 1 : non complété (renvoie null).");
}
else
{
    var cats = string.Join(", ", explicite.Categories);
    Console.WriteLine($"Catégories (enregistrement explicite) : {cats}");
}


Exercice 1 : non complété (renvoie null).



(2,26): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## Moment 2 — Sérialisation round-trip (JSON + XML)

Le même graphe d'entités doit pouvoir être **persisté** puis **reconstitué** sans perte
— c'est la promesse d'un socle. On construit une hiérarchie `Catalog > Product > Variant`
sur 3 niveaux, on la sérialise dans les deux formats, puis on désérialise et on vérifie
que (a) la structure est identique et (b) les références `Parent` sont re-liées.


In [5]:
// Construction du graphe : Catalog > Product > Variant (3 niveaux).
var catalogue = new Catalog { Nom = "Boutique" };

var logiciel = new Product { Nom = "Logiciel", Prix = 199 };
var matos    = new Product { Nom = "Matériel", Prix = 0 };
catalogue.AddChild(logiciel);
catalogue.AddChild(matos);

logiciel.AddChild(new Variant { Nom = "Licence Pro",  Prix = 299 });
logiciel.AddChild(new Variant { Nom = "Licence Basic", Prix = 99 });
matos.AddChild(new Variant { Nom = "Clé USB", Prix = 25 });

void Afficher(IChildEntity noeud, int prof = 0)
{
    var nom = noeud switch { Catalog c => $"Catalog:{c.Nom}", Product p => $"Product:{p.Nom}",
                             Variant v => $"Variant:{v.Nom}", _ => noeud.GetType().Name };
    var parent = noeud.Parent is null ? "(racine)" : noeud.Parent.GetType().Name;
    Console.WriteLine($"{new string(' ', prof * 2)}- {nom}  [parent: {parent}]");
    foreach (var enfant in noeud.Children) Afficher(enfant, prof + 1);
}

Console.WriteLine("Graphe original :");
Afficher(catalogue);


Graphe original :


- Catalog:Boutique  [parent: (racine)]


  - Product:Logiciel  [parent: Catalog]


    - Variant:Licence Pro  [parent: Product]


    - Variant:Licence Basic  [parent: Product]


  - Product:Matériel  [parent: Catalog]


    - Variant:Clé USB  [parent: Product]


In [6]:
// Round-trip JSON : sérialise -> désérialise -> compare la structure.
string json = MetadataJsonSerializer.Serialize(catalogue);
Console.WriteLine("JSON (extrait) :");
Console.WriteLine(json.Substring(0, Math.Min(240, json.Length)) + (json.Length > 240 ? "  ..." : ""));

var depuisJson = MetadataJsonSerializer.Deserialize<Catalog>(json);
Console.WriteLine();
Console.WriteLine("Graphe rechargé depuis JSON :");
Afficher(depuisJson);
Console.WriteLine();
Console.WriteLine($"Parent du 1er produit re-lié ? {depuisJson.Children[0].Parent?.GetType().Name == "Catalog"}");


JSON (extrait) :


{
  "Nom": "Boutique",
  "Children": {
    "$type": "System.Collections.Generic.List`1[[MyIA.AI.ComponentModel.Entities.IChildEntity, MyIA.AI.Shared]], System.Private.CoreLib",
    "$values": [
      {
        "$type": "Submission#3+P  ...


Graphe rechargé depuis JSON :


- Catalog:Boutique  [parent: (racine)]


  - Product:Logiciel  [parent: Catalog]


    - Variant:Licence Pro  [parent: Product]


    - Variant:Licence Basic  [parent: Product]


  - Product:Matériel  [parent: Catalog]


    - Variant:Clé USB  [parent: Product]


Parent du 1er produit re-lié ? True


In [7]:
// Round-trip XML : même graphe, autre format, même résultat.
// Le serializer XML a besoin du résolveur décoré avec les types concrets connus du graphe
// (la moitié A2 du socle : DynamicSurrogate + re-lien Parent via AddChild au rechargement).
var xmlSerializer = new MetadataXmlSerializer(
    new XmlAwareContractResolver(new[] { typeof(Catalog), typeof(Product), typeof(Variant) }));
string xml = xmlSerializer.Serialize(catalogue);
Console.WriteLine("XML (extrait) :");
Console.WriteLine(xml.Substring(0, Math.Min(260, xml.Length)) + (xml.Length > 260 ? "  ..." : ""));

var depuisXml = xmlSerializer.Deserialize<Catalog>(xml);
Console.WriteLine();
Console.WriteLine($"Nombre de produits rechargés depuis XML : {depuisXml.Children.Count}");
var premierProduit = depuisXml.Children.OfType<Product>().First();
Console.WriteLine($"Variantes du 1er produit : {premierProduit.Children.Count}");


XML (extrait) :


<?xml version="1.0" encoding="utf-16"?>
<NodeSurrogate xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" type="Catalog">
  <property name="Nom" value="Boutique" />
  <child type="Product">
    <property name  ...


Nombre de produits rechargés depuis XML : 2


Variantes du 1er produit : 2


### Exercice 2 — Sérialiser une hiérarchie personnalisée

**Objectif.** Construire un second catalogue contenant un seul produit `Formation` avec
deux variantes (`Présentiel`, `Distanciel`), le sérialiser en JSON, puis désérialiser et
renvoyer le **nom de la première variante** du premier produit.

*Indice.* Utilise `MetadataJsonSerializer.Serialize` puis `Deserialize<Catalog>`, et
accède à `.Children.OfType<Product>().First().Children` pour atteindre les variantes.


In [8]:
// Exercice 2 (a completer) : construire, sérialiser, recharger, extraire.
public string? NomPremiereVariante()
{
    // TODO etudiant : construire Catalog > Product("Formation") >
    //                  Variant("Présentiel"), Variant("Distanciel"),
    //                  sérialiser en JSON, désérialiser, renvoyer le nom
    //                  de la première variante du premier produit.
    return null;  // TODO etudiant
}

// --- zone de validation (ne pas modifier) ---
var rep2 = NomPremiereVariante();
Console.WriteLine(rep2 is null
    ? "Exercice 2 : non complété (renvoie null)."
    : $"Exercice 2 -> première variante : {rep2}");


Exercice 2 : non complété (renvoie null).



(2,14): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## Moment 3 — Prédicat universel métier (Flee)

C'est le moment où la logique métier **cesse d'être codée**. Une règle
métier (quels abonnements sont éligibles à un traitement) est une **chaîne** :

```
Montant > 1000 And Pays = "FR" And Actif = true
```

Cette chaîne peut venir d'un fichier de configuration, d'une ligne de CSV, ou être saisie
par un utilisateur non-développeur dans une console d'administration. On la **compile une fois**
en un délégué réutilisable, puis on l'évalue sur N instances en changeant
simplement les variables — **sans recompiler l'hôte**.

> **Le socle industrialise ce moment.** Plutôt que d'enregistrer chaque variable
> à la main (`ctx.Variables.Add(...)`), le socle expose
> [`FleePredicateBuilder.Create<T>(regle)`](../../../MyIA.AI.Shared/ComponentModel/Rules/FleePredicateBuilder.cs) :
> il reflète les propriétés publiques de `T` comme variables de la règle et renvoie
> un `Func<T, bool>` directement utilisable dans LINQ. La signature du type fait
> office de contrat de variables — l'erreur « variable non déclarée » disparaît.

> **Syntaxe Flee.** Le moteur est d'inspiration VB : `And` / `Or` / `=`, **pas**
> `&&` / `||` / `==`. La comparaison de chaînes utilise `=`.

In [9]:
// Des abonnements factices sur lesquels appliquer la règle.
var abonnements = new List<Subscription>
{
    new() { Name = "ENT-001", Montant = 1500, Pays = "FR", Actif = true   },
    new() { Name = "ENT-002", Montant =  500, Pays = "FR", Actif = true   },
    new() { Name = "ENT-003", Montant = 3000, Pays = "US", Actif = true   },
    new() { Name = "ENT-004", Montant = 2000, Pays = "FR", Actif = false  },
};

// La règle est une CHAÎNE — elle aurait aussi bien pu être lue dans un fichier.
string regle = "Montant > 1000 And Pays = \"FR\" And Actif = true";
Console.WriteLine($"Règle appliquée : {regle}");
Console.WriteLine();

// Compilation unique : le socle (FleePredicateBuilder) reflète automatiquement
// les propriétés publiques de Subscription (Montant, Pays, Actif) et les expose
// comme variables de la règle. Fini l'enregistrement manuel ctx.Variables.Add :
// la signature du type fait office de contrat de variables.
var predicat = FleePredicateBuilder.Create<Subscription>(regle);

Console.WriteLine("Eligibles :");
foreach (var ab in abonnements)
{
    Console.WriteLine($"  {ab.Name,-8} Montant={ab.Montant,6} Pays={ab.Pays} Actif={ab.Actif,-5} -> {predicat(ab)}");
}

// On change la règle (les données pilotent), toujours sans recompiler l'hôte :
// un nouveau Create<T> recompile un délégué indépendant. Les deux règles coexistent.
string regle2 = "Montant > 2000";
var predicat2 = FleePredicateBuilder.Create<Subscription>(regle2);
Console.WriteLine();
Console.WriteLine($"Règle 2 : {regle2}");
var gros = abonnements.Where(predicat2);
Console.WriteLine($"Gros abonnements (>2000) : {string.Join(", ", gros.Select(a => a.Name))}");

Règle appliquée : Montant > 1000 And Pays = "FR" And Actif = true


Eligibles :


  ENT-001  Montant=  1500 Pays=FR Actif=True  -> True


  ENT-002  Montant=   500 Pays=FR Actif=True  -> False


  ENT-003  Montant=  3000 Pays=US Actif=True  -> False


  ENT-004  Montant=  2000 Pays=FR Actif=False -> False


Règle 2 : Montant > 2000


Gros abonnements (>2000) : ENT-003


### Exercice 3 — Une règle à seuil variable

**Objectif.** Compléter `FiltrerParMontant(min)` pour qu'elle renvoie les noms des
abonnements dont le `Montant` est **supérieur ou égal** au seuil passé en paramètre.
La règle doit être **construite comme une chaîne** à partir du seuil (`$"Montant >= {min}"`),
compilée via le socle, puis appliquée.

*Indice.* `FleePredicateBuilder.Create<Subscription>($"Montant >= {min}")` renvoie
un `Func<Subscription, bool>` : passez-le directement à `.Where(...)` puis sélectionnez
les `Name`. Syntaxe Flee : `>=` pour « supérieur ou égal ».

In [10]:
// Exercice 3 (a completer) : règle de seuil variable, compilée depuis une chaîne.
public IEnumerable<string> FiltrerParMontant(IEnumerable<Subscription> abonnements, double min)
{
    // TODO etudiant : construire la règle "$"Montant >= {min}"", la compiler via Flee,
    //                  et renvoyer les noms des abonnements au-dessus du seuil.
    return Enumerable.Empty<string>();  // TODO etudiant
}

// --- zone de validation (ne pas modifier) ---
var retenus = FiltrerParMontant(abonnements, 1500);
Console.WriteLine(retenus.Any()
    ? $"Exercice 3 -> abonnements >= 1500 : {string.Join(", ", retenus)}"
    : "Exercice 3 : non complété (renvoie une liste vide).");


Exercice 3 : non complété (renvoie une liste vide).


## Conclusion

Trois moments, un même fil rouge : **séparer la structure de l'utilisation**.

| Moment | Ce qui change | Bénéfice |
|---|---|---|
| **Décoration → introspection** | On déclare le rôle d'un type par attribut | Un nouveau type est découvrable **sans toucher au code qui le consomme** |
| **Sérialisation round-trip** | Le graphe persiste et se reconstitue | La configuration peut être **un fichier**, pas du code |
| **Prédicat universel (Flee)** | La règle métier est une chaîne | La logique est pilotée par les **données**, éditable sans redéploiement |

Le socle `MyIA.AI.Shared` factorise ces trois besoins transverses (découverte, persistance,
logique déclarative) afin que les modules métier se concentrent sur leur domaine propre.
C'est le sens d'un *socle* : ce qui était ré-écrit dans chaque projet devient une
convention partagée, testée une fois pour toutes (48 tests unitaires au passage).
